# Normalizing flows — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def normal_pdf(x,mu,sigma):
    return np.exp(-0.5*((x-mu)/sigma)**2)/(sigma*np.sqrt(2*np.pi))
def softmax(a):
    a=np.asarray(a,dtype=float); e=np.exp(a-a.max()); return e/e.sum()
def mse(a,b): return float(np.mean((np.asarray(a)-np.asarray(b))**2))
print('setup ok')

## invertible density transformation
Change of variables from calculus is the core prerequisite, and neural networks provide flexible invertible maps. Flows contrast with VAEs (10.2), GANs (10.5), and diffusion (10.12) because they keep exact likelihoods. They also foreshadow flow matching (10.16).

In [ ]:
# 1) affine flow maps base samples and tracks log determinant
x=np.random.default_rng(0).normal(size=(200,2)); a=1.5; t=np.array([.3,-.2]); y=a*x+t
plt.figure(figsize=(4,4)); plt.scatter(y[:,0],y[:,1],s=8); plt.axis('equal'); plt.title('affine flow samples')
assert abs(2*math.log(a)-0.8109302162)<1e-9

In [ ]:
# 2) inverse recovers the base point exactly
x0=np.array([1.2,-.7]); y0=a*x0+t; back=(y0-t)/a
plt.figure(figsize=(4,4)); plt.plot([x0[0],y0[0]],[x0[1],y0[1]],'o-'); plt.title('forward and inverse')
assert np.allclose(back,x0)

In [ ]:
# 3) stretching lowers density by log determinant
base_log=-0.5*np.sum(x0**2)-math.log(2*math.pi); flow_log=base_log-2*math.log(a)
plt.figure(figsize=(4,3)); plt.bar(['base logp','flow logp'],[base_log,flow_log]); plt.title('change of variables')
assert flow_log<base_log

In [ ]:
# 4) a coupling layer changes one coordinate using the other
x=np.random.default_rng(1).normal(size=(200,2)); y=x.copy(); y[:,1]=x[:,1]+0.8*np.tanh(x[:,0])
plt.figure(figsize=(4,4)); plt.scatter(y[:,0],y[:,1],s=8); plt.title('coupling layer bends density')
assert y.shape==x.shape

In [ ]:
# 5) composing invertible layers stays invertible
z=(y-t)/a; y2=a*z+t
plt.figure(figsize=(4,4)); plt.scatter(y[:,0],y[:,1],s=8,label='original'); plt.scatter(y2[:,0],y2[:,1],s=3,label='roundtrip'); plt.legend(); plt.title('round-trip check')
assert np.allclose(y2,y)